# Build web assets for the San Antonio bicyclist scrolly

This notebook is the bridge between the analysis repository and the separate custom scrolly repository. It reads the validated TxDOT CRIS export and local street centerlines, then writes small, deployment-ready files to `outputs/web_data/`.

The reporting window is Jan. 1, 2024 through Sept. 1, 2026. The expected audit is 82 qualifying San Antonio crashes, 83 bicyclists, 13 deaths and 70 suspected serious injuries. Two qualifying crashes have no usable coordinates, so the map contains 80 points. The nine candidate corridors are exploratory repeat-crash stretches, not an official City ranking.


In [ ]:
from pathlib import Path
import json
import zipfile

import pandas as pd
import geopandas as gpd
from sklearn.cluster import AgglomerativeClustering

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'outputs'
WEB = OUT / 'web_data'
WEB.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load and filter the person-level CRIS export.
raw = pd.read_csv(RAW / 'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[
    (raw['City'] == 'SAN ANTONIO') &
    (raw['Person Type'] == '3 - PEDALCYCLIST') &
    raw['Person Injury Severity'].isin(severity)
].copy()
target['date'] = pd.to_datetime(target['Crash Date'], errors='coerce')
target = target[target['date'].between('2024-01-01', '2026-09-01')].copy()
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)

factor_cols = ['Contributing Factor 1', 'Contributing Factor 2', 'Contributing Factor 3']
def combine_values(values):
    return '; '.join(sorted({str(v).strip() for v in values.dropna()
                             if str(v).strip() and str(v).strip().upper() not in {'NAN', 'NONE', 'NO DATA'}}))

target['factors'] = target[factor_cols].fillna('').astype(str).agg('; '.join, axis=1)
crashes = target.groupby('Crash ID', as_index=False).agg(
    date=('date', 'first'),
    year=('Crash Year', 'first'),
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first'),
    deaths=('death', 'sum'),
    serious_injuries=('serious_injury', 'sum'),
    victim_ages=('Person Age', combine_values),
    victim_genders=('Person Gender', combine_values),
    victim_helmets=('Person Helmet', combine_values),
    contributing_factors=('factors', combine_values),
    street_name=('Street Name', 'first'),
    intersecting_street=('Intersecting Street Name', 'first'),
)
crashes['year'] = pd.to_numeric(crashes['year'], errors='coerce').astype('Int64')
points = gpd.GeoDataFrame(
    crashes.dropna(subset=['latitude', 'longitude']).copy(),
    geometry=gpd.points_from_xy(crashes.dropna(subset=['latitude', 'longitude'])['longitude'], crashes.dropna(subset=['latitude', 'longitude'])['latitude']),
    crs=4326,
)
assert len(crashes) == 82, f'Expected 82 crashes; got {len(crashes)}'
assert len(points) == 80, f'Expected 80 mappable points; got {len(points)}'
assert int(target.death.sum()) == 13, f'Expected 13 deaths; got {target.death.sum()}'
assert int(target.serious_injury.sum()) == 70, f'Expected 70 serious injuries; got {target.serious_injury.sum()}'
print(f'{len(crashes)} crashes | {len(target)} bicyclists | {int(target.death.sum())} deaths | {int(target.serious_injury.sum())} serious injuries | {len(points)} mapped points')


In [ ]:
# Match crashes to local street segments and reproduce the nine exploratory corridors.
streets_dir = RAW / 'streets'
street_shp = streets_dir / 'Streets' / 'Streets.shp'
if not street_shp.exists():
    with zipfile.ZipFile(RAW / 'Streets.zip') as archive:
        archive.extractall(streets_dir)

roads = gpd.read_file(street_shp).rename(columns={
    'CartID': 'segmentid', 'MSAG_NAME': 'road_label',
    'FROM_STREE': 'from_street', 'TO_STREET': 'to_street', 'CoSARoadFu': 'road_class'
})
roads['length_miles'] = pd.to_numeric(roads['LengthFeet'], errors='coerce') / 5280
roads = roads[roads['length_miles'] > 0].copy().to_crs(2279)
fields = ['segmentid', 'road_label', 'from_street', 'to_street', 'road_class', 'length_miles', 'geometry']
matches = gpd.sjoin_nearest(points.to_crs(roads.crs), roads[fields], how='left', distance_col='match_distance_ft')
matches = matches[matches['match_distance_ft'] <= 150].sort_values('match_distance_ft').drop_duplicates('Crash ID').copy()

matches['candidate_group'] = pd.NA
group_number = 0
for road_name, group in matches.groupby('road_label', dropna=False):
    if len(group) < 2 or pd.isna(road_name) or not str(road_name).strip():
        continue
    coords = [[p.x, p.y] for p in group.geometry]
    labels = AgglomerativeClustering(n_clusters=None, distance_threshold=3 * 5280, linkage='complete').fit_predict(coords)
    for label in sorted(set(labels)):
        idx = group.index[labels == label]
        if len(idx) >= 2:
            matches.loc[idx, 'candidate_group'] = group_number
            group_number += 1
matches = matches[matches['candidate_group'].notna()].copy()

def join_values(values):
    return ', '.join(sorted({str(v).strip() for v in values.dropna()
                             if str(v).strip() and str(v).strip().upper() not in {'NAN', 'NONE', 'NO DATA'}}))

corridors = matches.groupby('candidate_group', as_index=False).agg(
    crashes=('Crash ID', 'nunique'), deaths=('deaths', 'sum'),
    serious_injuries=('serious_injuries', 'sum'), first_year=('year', 'min'),
    last_year=('year', 'max'), roads=('road_label', join_values),
    from_streets=('from_street', join_values), to_streets=('to_street', join_values),
    max_match_distance_ft=('match_distance_ft', 'max')
)
spans = []
for gid, group in matches.groupby('candidate_group'):
    spans.append({'candidate_group': gid, 'span_miles': max(group.geometry.x.max() - group.geometry.x.min(), group.geometry.y.max() - group.geometry.y.min()) / 5280})
corridors = corridors.merge(pd.DataFrame(spans), on='candidate_group', how='left')
corridors = corridors.sort_values(['crashes', 'deaths', 'serious_injuries'], ascending=False).reset_index(drop=True)
corridors['display_order'] = range(1, len(corridors) + 1)
assert len(corridors) == 9, f'Expected 9 corridors; got {len(corridors)}'
print(corridors[['display_order', 'roads', 'crashes', 'deaths', 'serious_injuries']].to_string(index=False))


In [ ]:
# Write compact GeoJSON and JSON assets for the custom build repo.
def clean(value):
    if pd.isna(value):
        return ''
    if hasattr(value, 'item'):
        value = value.item()
    return value

crash_features = []
for _, row in points.iterrows():
    group = matches.loc[matches['Crash ID'] == row['Crash ID'], 'candidate_group']
    crash_features.append({
        'type': 'Feature',
        'properties': {
            'kind': 'crash', 'crash_id': str(row['Crash ID']), 'year': int(row['year']),
            'deaths': int(row['deaths']), 'serious_injuries': int(row['serious_injuries']),
            'victim_ages': clean(row['victim_ages']), 'victim_genders': clean(row['victim_genders']),
            'victim_helmets': clean(row['victim_helmets']), 'contributing_factors': clean(row['contributing_factors']),
            'street_name': clean(row['street_name']), 'intersecting_street': clean(row['intersecting_street']),
            'candidate_group': int(group.iloc[0]) if len(group) else None,
        },
        'geometry': {'type': 'Point', 'coordinates': [float(row.geometry.x), float(row.geometry.y)]}
    })

segment_groups = matches[['segmentid', 'candidate_group']].dropna().drop_duplicates()
candidate_lines = roads.merge(segment_groups, on='segmentid', how='inner').dissolve(by='candidate_group', as_index=False)
line_geo = candidate_lines.to_crs(4326)
line_features = []
for _, row in line_geo.iterrows():
    geometry = json.loads(gpd.GeoSeries([row.geometry], crs=4326).to_json())['features'][0]['geometry']
    summary = corridors[corridors['candidate_group'] == row['candidate_group']].iloc[0].to_dict()
    props = {k: clean(v) for k, v in summary.items()}
    props.update({'kind': 'corridor', 'candidate_group': int(row['candidate_group'])})
    line_features.append({'type': 'Feature', 'properties': props, 'geometry': geometry})

def write_geojson(path, features):
    path.write_text(json.dumps({'type': 'FeatureCollection', 'features': features}, separators=(',', ':')))

write_geojson(WEB / 'crashes.geojson', crash_features)
write_geojson(WEB / 'corridors.geojson', line_features)

stats = {
    'reporting_window': 'Jan. 1, 2024–Sept. 1, 2026',
    'crashes': int(len(crashes)), 'mapped_points': int(len(points)),
    'unmapped_crashes': int(len(crashes) - len(points)),
    'bicyclists': int(target.shape[0]), 'deaths': int(target.death.sum()),
    'serious_injuries': int(target.serious_injury.sum()),
    'corridors': int(len(corridors)), 'source': 'TxDOT CRIS',
    'definition': 'Pedalcyclists with fatal or suspected serious injury in San Antonio crashes.',
    'corridor_note': 'Exploratory repeat-crash stretches; not an official City ranking.'
}
(WEB / 'stats.json').write_text(json.dumps(stats, indent=2))
corridors.to_csv(WEB / 'corridors.csv', index=False)
print('Wrote crashes.geojson, corridors.geojson, stats.json and corridors.csv')


In [ ]:
# Final audit for deployment.
assert json.loads((WEB / 'crashes.geojson').read_text())['features'].__len__() == 80
assert json.loads((WEB / 'corridors.geojson').read_text())['features'].__len__() == 9
assert json.loads((WEB / 'stats.json').read_text())['deaths'] == 13
print('PASS: web assets match the analysis totals.')
